In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("comm_log.db")
conn

In [2]:
query = "SELECT * FROM communication_log LIMIT 5;"
df = pd.read_sql_query(query, conn)
df

,id,merchant_id,communication_id,customer_id,communication_type,delivery_status,sent_time,scheduled_time,credit_used,channel
0,1,501,9001,C1,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
1,2,501,9001,C2,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
2,3,501,9002,C2,2,900,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
3,4,501,9001,C3,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
4,5,501,9002,C3,2,1100,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms


In [3]:
query = "SELECT * FROM campaign;"
df = pd.read_sql_query(query, conn)
df

,id,merchant_id,parent_id,name,creation_status,processing_status
0,9001,501,NaN,Diwali Cart Recovery - Wave 1,approved,processed
1,9002,501,9001.0,Diwali Cart Recovery - Retry A,approved,processed
2,9003,501,9002.0,Diwali Cart Recovery - Retry B,approved,processed
3,9004,501,9001.0,Diwali Cart Recovery - Retry C (pending),approval_awaiting,processed
4,9101,501,NaN,Diwali Flash Sale - Standalone,approved,processed
5,9201,501,NaN,Diwali Wave 2,approved,processed
6,9202,501,9201.0,Diwali Wave 2 - Retry,approved,processed


In [4]:
query = "SELECT COUNT(*) AS total_sends FROM communication_log;"
df = pd.read_sql_query(query, conn)
df

,total_sends
0,30


In [5]:
query = "SELECT COUNT(DISTINCT customer_id) AS distinct_customers FROM communication_log;"
df = pd.read_sql_query(query, conn)
df

,distinct_customers
0,25


In [6]:
query = """
SELECT *
FROM campaign
WHERE creation_status = 'approval_awaiting';
"""
df = pd.read_sql_query(query, conn)
df

,id,merchant_id,parent_id,name,creation_status,processing_status
0,9004,501,9001,Diwali Cart Recovery - Retry C (pending),approval_awaiting,processed


In [7]:
query = """
SELECT COUNT(*) AS sends
FROM communication_log
WHERE communication_id = 9004;
"""
df = pd.read_sql_query(query, conn)
df

,sends
0,4


In [8]:
query = """
SELECT processing_status, COUNT(*) AS campaigns
FROM campaign
GROUP BY processing_status;
"""
df = pd.read_sql_query(query, conn)
df

,processing_status,campaigns
0,processed,7


In [9]:
query = """
SELECT delivery_status, COUNT(*) AS sends
FROM communication_log
GROUP BY delivery_status;
"""
df = pd.read_sql_query(query, conn)
df

,delivery_status,sends
0,900,26
1,1100,4


In [10]:
query = """
SELECT COUNT(*) AS target_base
FROM communication_log cl
JOIN campaign c
    ON cl.communication_id = c.id
WHERE c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
  AND c.processing_status = 'processed'
  AND cl.delivery_status = 900;
"""
df = pd.read_sql_query(query, conn)
df

,target_base
0,22


In [11]:
query = """
WITH RECURSIVE campaign_tree AS (
    SELECT
        id AS campaign_id,
        id AS root_id
    FROM campaign
    WHERE parent_id IS NULL

    UNION ALL

    SELECT
        c.id,
        ct.root_id
    FROM campaign c
    JOIN campaign_tree ct
        ON c.parent_id = ct.campaign_id
)
SELECT
    root_id,
    campaign_id
FROM campaign_tree
ORDER BY root_id, campaign_id;
"""
df = pd.read_sql_query(query, conn)
df

,root_id,campaign_id
0,9001,9001
1,9001,9002
2,9001,9003
3,9001,9004
4,9101,9101
5,9201,9201
6,9201,9202


In [16]:
query = """
WITH RECURSIVE chains AS (
    SELECT id AS campaign_id, id AS root_id
    FROM campaign
    WHERE parent_id IS NULL

    UNION ALL

    SELECT c.id, chains.root_id
    FROM campaign c
    JOIN chains
        ON c.parent_id = chains.campaign_id
),

sends AS (
    SELECT
        chains.root_id,
        cl.id AS send_id,
        cl.customer_id
    FROM communication_log cl
    JOIN chains
        ON cl.communication_id = chains.campaign_id
    JOIN campaign c
        ON c.id = cl.communication_id
    WHERE c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
      AND c.processing_status = 'processed'
      AND cl.delivery_status = 900
)

SELECT
    root_id,
    COUNT(DISTINCT customer_id) AS target_base
FROM sends
WHERE root_id IN (9001, 9201)
GROUP BY root_id

UNION ALL

SELECT
    root_id,
    COUNT(*) AS target_base
FROM sends
WHERE root_id = 9101
GROUP BY root_id;
"""
df = pd.read_sql_query(query, conn)
df

,root_id,target_base
0,9001,10
1,9201,5
2,9101,7


In [17]:
query = """
WITH RECURSIVE chains AS (
    SELECT id AS campaign_id, id AS root_id
    FROM campaign
    WHERE parent_id IS NULL

    UNION ALL

    SELECT c.id, chains.root_id
    FROM campaign c
    JOIN chains
        ON c.parent_id = chains.campaign_id
),

sends AS (
    SELECT
        chains.root_id,
        cl.id AS send_id,
        cl.customer_id
    FROM communication_log cl
    JOIN chains
        ON cl.communication_id = chains.campaign_id
    JOIN campaign c
        ON c.id = cl.communication_id
    WHERE c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
      AND c.processing_status = 'processed'
      AND cl.delivery_status = 900
),

counts AS (
    -- Retry chains: one customer counted once
    SELECT
        root_id,
        COUNT(DISTINCT customer_id) AS target_base
    FROM sends
    WHERE root_id IN (9001, 9201)
    GROUP BY root_id

    UNION ALL

    -- Standalone communication: every send is an event
    SELECT
        root_id,
        COUNT(*) AS target_base
    FROM sends
    WHERE root_id = 9101
    GROUP BY root_id
)

SELECT SUM(target_base) AS target_base
FROM counts;
"""

df = pd.read_sql_query(query, conn)
df

,target_base
0,22
